# Training IndoBERT Sentiment Analysis (3 Classes) - GOOGLE COLAB VERSION

## Objectives
- **Model:** `indobenchmark/indobert-base-p1`
- **Task:** Sentiment Classification (3 Classes: Negative, Neutral, Positive)
- **Optimization:** GPU (Mixed Precision fp16), Early Stopping, Best Model Loading
- **Target:** High Accuracy without Overfitting

**Note:** This notebook is optimized for Google Colab environment. 
**Setup:** Runtime → Change runtime type → GPU (T4 recommended)

In [ ]:
# 1. Setup & Imports
!pip install -q transformers datasets scikit-learn accelerate pandas seaborn matplotlib

import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Check GPU
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device_name}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# 2. Load Data & Preprocessing
# Upload file CSV ke Colab
from google.colab import files
import io

print("📤 Upload file CSV Anda (gojek_scraped_3class_RELABELED.csv)...")
uploaded = files.upload()

# Get filename
file_path = list(uploaded.keys())[0]
print(f"✅ File berhasil di-upload: {file_path}")

df = pd.read_csv(file_path)
print(f"Total baris: {len(df)}")

# Label Mapping
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Split Data (Stratified to keep balance)
# 80% Train, 10% Validation, 10% Test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, 
    temp_labels, 
    test_size=0.5, 
    stratify=temp_labels, 
    random_state=42
)

print(f"Train Size: {len(train_texts)}")
print(f"Val Size:   {len(val_texts)}")
print(f"Test Size:  {len(test_texts)}")

In [ ]:
# 3. Tokenization
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    dataset = []
    for i in range(len(texts)):
        item = {key: torch.tensor(val[i]) for key, val in encodings.items()}
        item['labels'] = torch.tensor(labels[i])
        dataset.append(item)
    return dataset

train_dataset = tokenize_data(train_texts, train_labels)
val_dataset = tokenize_data(val_texts, val_labels)
test_dataset = tokenize_data(test_texts, test_labels)

In [ ]:
# 4. Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# 5. Training Configuration
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3, 
    id2label=id2label, 
    label2id=label2id
)

# TRAINING ARGUMENTS (Optimized for Google Colab)
training_args = TrainingArguments(
    output_dir='./results_3class_relabeled',
    num_train_epochs=5,              # Max epochs (Early stopping will likely stop earlier)
    per_device_train_batch_size=16,  # Standard for Colab T4 (16GB VRAM)
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,   # Virtual batch size = 32
    learning_rate=2e-5,              # Low LR to prevent overfitting
    weight_decay=0.01,               # Regularization
    warmup_ratio=0.1,
    eval_strategy="epoch",           # Check val every epoch
    save_strategy="epoch",           # Save model every epoch
    load_best_model_at_end=True,     # ALWAYS load the best model found
    metric_for_best_model="accuracy",
    fp16=True,                       # GPU Acceleration (Mixed Precision)
    logging_dir='./logs',
    logging_steps=10,                # LEBIH SERING LOG (setiap 10 step)
    logging_first_step=True,         # LOG STEP PERTAMA
    dataloader_num_workers=2,        # Colab supports multiprocessing
    disable_tqdm=False               # TAMPILKAN PROGRESS BAR
)

print("✅ Model & Training Config ready!")

In [ ]:
# 6. Initialize Trainer & Start Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if no improve for 2 epochs
)

# START TRAINING
print("Starting training...")
trainer.train()

In [ ]:
# 7. Evaluation on Test Set
print("Evaluating on Test Set...")
test_result = trainer.predict(test_dataset)
print(test_result.metrics)

In [ ]:
# 8. Confusion Matrix & Classification Report
y_preds = np.argmax(test_result.predictions, axis=1)
y_true = test_result.label_ids

print("\n📋 Classification Report:")
print(classification_report(y_true, y_preds, target_names=['negative', 'neutral', 'positive']))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['negative', 'neutral', 'positive'], 
            yticklabels=['negative', 'neutral', 'positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - IndoBERT 3-Class Sentiment')
plt.show()

In [ ]:
# 9. Save Final Model
save_path = './saved_model_indobert_3class_relabeled'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Create zip for easy download
import shutil
shutil.make_archive("model_3class_relabeled", 'zip', save_path)
print("✅ Model zipped as model_3class_relabeled.zip")

# Download zip file
from google.colab import files
files.download('model_3class_relabeled.zip')
print("📥 Download started!")

## 🎉 Training Complete!

### Expected Results:
- **Accuracy**: 88-95%
- **F1-Score**: >0.85
- **Model**: saved_model_indobert_3class_relabeled.zip

### Download Model:
Model akan otomatis terdownload ke komputer Anda dalam format ZIP.

### Tips:
- Jika accuracy rendah, coba tambah data training
- Jika overfitting, turunkan learning rate atau tambah regularization
- Monitor confusion matrix untuk identify weak classes

### Google Colab Notes:
- Pastikan GPU aktif: Runtime → Change runtime type → GPU
- Session timeout ~12 jam (free tier)
- Jika disconnect, model checkpoint tersimpan di `./results_3class_relabeled/`